##### Copyright 2026 Google LLC.


In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# Gemini API: Deep Reasoning and Automated Code Security Auditing with Gemini 4 Argon


<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Deep_Reasoning_and_Code_Auditing_with_Gemini_4_Argon.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/>
</a>


<!-- Community Contributor Badge -->
<table>
  <tr>
    <!-- Author Avatar Cell -->
    <td bgcolor="#d7e6ff">
      <a href="https://github.com/nandhakumar-murugan" target="_blank" title="View Nandhakumar's profile on GitHub">
        <img src="https://github.com/nandhakumar-murugan.png?size=100"
             alt="Nandhakumar Murugan's GitHub avatar"
             width="100"
             height="100">
      </a>
    </td>
    <!-- Text Content Cell -->
    <td bgcolor="#d7e6ff">
      <h2><font color='black'>This notebook was contributed by <a href="https://github.com/nandhakumar-murugan" target="_blank"><font color='#217bfe'><strong>Nandhakumar Murugan</strong></font></a>.</font></h2>
      <h5><font color='black'><a href="https://www.linkedin.com/in/nandhakumar-murugan"><font color="#078efb">Find me on LinkedIn</font></a> - Google Student Ambassador & AI Engineer - See <a href="https://github.com/nandhakumar-murugan" target="_blank"><font color="#078efb"><strong>Nandhakumar's</strong></font></a> other contributions <a href="https://github.com/search?q=repo%3Agoogle-gemini%2Fcookbook+%22nandhakumar-murugan%22&type=code" target="_blank"><font color="#078efb">here</font></a>.</font></h5><br>
      <!-- Footer -->
      <font color='black'><small><em>Have a cool Gemini example? Feel free to <a href="https://github.com/google-gemini/cookbook/blob/main/CONTRIBUTING.md" target="_blank"><font color="#078efb">share it too</font></a>!</em></small></font>
    </td>
  </tr>
</table>


## Overview

Modern software security vulnerabilities rarely live in isolated functions. Instead, architectural flaws—such as **Time-of-Check to Time-of-Use (TOCTOU) race conditions**, broken object-level authorization, and subtle state synchronization bugs—span multiple interconnected modules. Auditing such issues requires deep reasoning across call boundaries rather than simple static pattern matching.

In this tutorial, you will explore **Gemini 4 Argon** (`gemini-4-argon`), Google's flagship reasoning model, and learn how to harness its tunable reasoning capabilities to perform multi-file code auditing and deterministic patch synthesis.

### What you will learn:
1. **Configuring Tunable Reasoning (`thinking_level`):** How to select between `LOW`, `MEDIUM`, and `HIGH` reasoning levels, and how reasoning budgets shape analysis depth.
2. **Inspecting Thought Signatures:** Capturing and reviewing the model's internal reasoning traces (`include_thoughts=True`) to audit how the model deduced the vulnerability.
3. **Multi-File Context Ingestion:** Ingesting interconnected Python modules within a single unified context window.
4. **Deterministic Structured Remediation:** Enforcing strict Pydantic `response_schema` to produce machine-verifiable `VulnerabilityAuditReport` objects with CWE mappings and unified diff patches.
5. **Automated Patch Application & Verification:** Programmatically applying the synthesized unified diff patch to verify the remediation locally.


## Setup

### Install the Google GenAI SDK

Install the official `google-genai` SDK and `pydantic` for structured data modeling.


In [1]:
%pip install -U -q "google-genai>=2.30.0" pydantic


Note: you may need to restart the kernel to use updated packages.


### Authenticate & Select Model

To use the Gemini API, you need an API key. If you don't already have one, create one in [Google AI Studio](https://aistudio.google.com/).

In Colab, store your key as a secret named `GEMINI_API_KEY`. Otherwise, ensure `GEMINI_API_KEY` is set in your environment.

Select the model using the parameter selector below.


In [2]:
import os
from google import genai
from google.genai import types

# Fetch API key from Colab secrets or environment variable
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except ImportError:
    api_key = os.environ.get('GEMINI_API_KEY')

if not api_key:
    raise ValueError("GEMINI_API_KEY is missing. Please set it in your environment or Colab secrets.")

client = genai.Client(api_key=api_key)
print("Gemini client successfully initialized!")

MODEL_ID = "gemini-4-argon"  # @param ["gemini-4-argon", "gemini-3.1-pro-preview", "gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash", "gemini-3.5-flash-lite", "gemini-2.5-pro"] {"allow-input": true, "isTemplate": true}
print(f"Active Model: {MODEL_ID}")


Gemini client successfully initialized!
Active Model: gemini-4-argon


## Define Synthetic Multi-File Architecture Fixture

To demonstrate genuine multi-file reasoning, define a realistic, synthetic micro-banking service across three files:
1. `session_store.py`: In-memory session and idempotency store with non-atomic state updates.
2. `auth_middleware.py`: Session extraction and token verification middleware.
3. `transfer_api.py`: Financial transaction endpoint that coordinates between authentication and session management.

Individually, each file appears standard and compiles without errors. However, an architectural **TOCTOU race condition (CWE-367)** exists across the boundary between `transfer_api.py` and `session_store.py`: the idempotency lock check and state modification are decoupled, allowing concurrent transfers to bypass debit balance verification under asynchronous load.


In [3]:
files_context = {
    "session_store.py": '''"""Session & Idempotency Store (Synthetic micro-service component)"""
import time
from typing import Dict, Any, Optional

class SessionStore:
    def __init__(self):
        self._sessions: Dict[str, Dict[str, Any]] = {}
        self._processed_keys: Dict[str, float] = {}

    def get_session(self, token: str) -> Optional[Dict[str, Any]]:
        return self._sessions.get(token)

    def set_session(self, token: str, data: Dict[str, Any]) -> None:
        self._sessions[token] = data

    def is_idempotency_key_active(self, key: str) -> bool:
        # Check if the transaction key was registered within 60s
        ts = self._processed_keys.get(key)
        if ts and (time.time() - ts) < 60.0:
            return True
        return False

    def mark_key_processed(self, key: str) -> None:
        # Non-atomic write without locking or atomic check-and-set
        self._processed_keys[key] = time.time()
''',

    "auth_middleware.py": '''"""Authentication Middleware"""
from typing import Optional, Dict, Any
from session_store import SessionStore

class AuthMiddleware:
    def __init__(self, session_store: SessionStore):
        self.session_store = session_store

    def authenticate_request(self, auth_header: Optional[str]) -> Optional[Dict[str, Any]]:
        if not auth_header or not auth_header.startswith("Bearer "):
            return None
        token = auth_header.split(" ")[1]
        session = self.session_store.get_session(token)
        if not session or not session.get("is_active", False):
            return None
        return session
''',

    "transfer_api.py": '''"""Funds Transfer API Endpoint"""
from typing import Dict, Any
from auth_middleware import AuthMiddleware
from session_store import SessionStore

class TransferService:
    def __init__(self, session_store: SessionStore, auth: AuthMiddleware):
        self.session_store = session_store
        self.auth = auth
        self.accounts: Dict[str, float] = {"user_1": 1000.0, "user_2": 250.0}

    def execute_transfer(self, auth_header: str, idempotency_key: str, recipient: str, amount: float) -> Dict[str, Any]:
        user_session = self.auth.authenticate_request(auth_header)
        if not user_session:
            return {"status": "error", "message": "Unauthorized"}

        sender = user_session["user_id"]

        # Flaw: Multi-file TOCTOU Race Condition (CWE-367)
        # There is a time gap between checking idempotency and marking it processed.
        # Concurrent async threads can pass this check before the key is recorded.
        if self.session_store.is_idempotency_key_active(idempotency_key):
            return {"status": "error", "message": "Duplicate transaction detected"}

        if self.accounts.get(sender, 0.0) < amount:
            return {"status": "error", "message": "Insufficient funds"}

        # Simulate network or database latency window
        # In multi-threaded execution, duplicate requests pass here!
        self.accounts[sender] -= amount
        self.accounts[recipient] = self.accounts.get(recipient, 0.0) + amount

        # Key is marked AFTER balance is deducted
        self.session_store.mark_key_processed(idempotency_key)

        return {"status": "success", "new_balance": self.accounts[sender]}
'''
}

print(f"Loaded synthetic micro-service with {len(files_context)} interconnected files.")


Loaded synthetic micro-service with 3 interconnected files.


## Define Structured Output Schema with Pydantic

To make the security audit actionable in automated CI/CD pipelines, enforce a strict schema using Pydantic. Gemini 4 Argon will return a structured, validated JSON payload containing:
* **CWE Identifier & Title** (e.g., `CWE-367: Time-of-Check Time-of-Use Race Condition`).
* **Multi-File Impact Chain**: Which files participate in the exploit window.
* **Root Cause & Exploitation Analysis**.
* **Unified Diff Patch**: A drop-in standard patch syntax that can be programmatically verified.


In [ ]:
from pydantic import BaseModel, Field
from typing import List

class VulnerabilityIssue(BaseModel):
    cwe_id: str = Field(description="The Common Weakness Enumeration ID, e.g. CWE-367")
    title: str = Field(description="Short descriptive title of the vulnerability")
    severity: str = Field(description="Severity rating: Critical, High, Medium, or Low")
    affected_files: List[str] = Field(description="List of files involved in the vulnerability chain")
    vulnerability_explanation: str = Field(description="Step-by-step breakdown of how the flaw can be exploited across files")
    root_cause_analysis: str = Field(description="Underlying algorithmic or architectural reason for the flaw")
    unified_diff_patch: str = Field(description="Standard unified diff format patch (diff -u) that remediates the vulnerability")

class VulnerabilityAuditReport(BaseModel):
    project_name: str = Field(description="Name of the audited project or module")
    summary: str = Field(description="Executive summary of the multi-file security audit")
    issues: List[VulnerabilityIssue] = Field(description="List of identified architectural and logic vulnerabilities")
    verification_steps: List[str] = Field(description="Actionable steps to verify that the patch prevents recurrence")


## Configure Tunable Reasoning (`thinking_level="HIGH"`)

Gemini 4 Argon supports tunable reasoning levels via `types.ThinkingConfig`:
* **`LOW` / `MINIMAL`:** Optimized for high-throughput, low-latency tasks where basic deduction suffices.
* **`MEDIUM`:** Balanced mode suitable for single-file refactoring and unit test generation.
* **`HIGH`:** Maximum reasoning budget. The model performs extensive internal chain-of-thought verification, exploring cross-file call traces, checking edge cases, and verifying mathematical or concurrency invariants before generating tokens.

By setting `include_thoughts=True`, you instruct the API to return the model's reasoning thought trace alongside the final response.


In [5]:
config = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(
        thinking_level="HIGH",
        include_thoughts=True
    ),
    response_mime_type="application/json",
    response_schema=VulnerabilityAuditReport,
    temperature=0.2,
)

print("GenerateContentConfig configured with thinking_level='HIGH' and structured schema.")


GenerateContentConfig configured with thinking_level='HIGH' and structured schema.


## Execute Multi-File Code Audit with Gemini 4 Argon

Format the codebase with clear file delimiters and submit it to the model using `MODEL_ID`.


In [6]:
# Format codebase context with explicit file boundaries
formatted_code_prompt = "You are a Principal Software Security Engineer. Conduct a rigorous security audit of this multi-file Python service.\n\n"

for filename, content in files_context.items():
    formatted_code_prompt += f"--- BEGIN FILE: {filename} ---\n{content}\n--- END FILE: {filename} ---\n\n"

formatted_code_prompt += (
    "Audit Instructions:\n"
    "1. Trace cross-file data flows and identify any concurrency, race conditions, or state synchronization vulnerabilities.\n"
    "2. For each vulnerability found, pinpoint the exact root cause and formulate an atomic remediation.\n"
    "3. Provide a production-grade unified diff patch that resolves the issue cleanly across the affected files."
)

print(f"Submitting multi-file context to {MODEL_ID} for deep reasoning audit...")
response = client.models.generate_content(
    model=MODEL_ID,
    contents=formatted_code_prompt,
    config=config
)
print("Audit analysis complete!")


Submitting multi-file context to gemini-4-argon for deep reasoning audit...
Audit analysis complete!


## Inspect Thought Traces & Reasoning Steps

Inspect the model's internal reasoning signature. When `include_thoughts=True` is enabled, the model's candidate parts contain thought blocks documenting its internal deduction steps before synthesizing the final output.


In [7]:
# Extract and inspect internal thoughts
thought_traces = []
for candidate in response.candidates:
    for part in candidate.content.parts:
        if getattr(part, 'thought', False):
            thought_traces.append(part.text)

print(f"Captured {len(thought_traces)} thought segment(s) from Gemini 4 Argon.\n")
if thought_traces:
    print("=" * 70)
    print("🧠 MODEL REASONING TRACE (Internal Chain-of-Thought):")
    print("=" * 70)
    preview_thought = thought_traces[0][:1200]
    print(preview_thought)
    if len(thought_traces[0]) > 1200:
        print("\n... [Reasoning trace continues with full concurrency path evaluation] ...")
    print("=" * 70)
else:
    print("Note: Thought traces may vary by environment and client configuration.")


Captured 1 thought segment(s) from Gemini 4 Argon.

🧠 MODEL REASONING TRACE (Internal Chain-of-Thought):
The user wants a rigorous security audit of a 3-file Python banking service.
Let's trace the cross-file interactions:
1. session_store.py:
   - SessionStore maintains _sessions and _processed_keys.
   - is_idempotency_key_active checks: ts and (time.time() - ts) < 60.0.
   - mark_key_processed writes: self._processed_keys[key] = time.time().
   - Notice that both methods operate without any thread locks or atomic synchronization.
2. auth_middleware.py:
   - Authenticates Bearer tokens against session_store. Standard dictionary lookup.
3. transfer_api.py:
   - execute_transfer flow:
     a. Authenticate sender via AuthMiddleware.
     b. Call self.session_store.is_idempotency_key_active(idempotency_key).
     c. Check balance: self.accounts.get(sender, 0.0) < amount.
     d. Deduct balance from sender, credit recipient.
     e. Call self.session_store.mark_key_processed(idempotency_k

## Parse & Review Structured Audit Findings

Because you enforced `response_schema=VulnerabilityAuditReport`, you can parse the response into the Pydantic model directly, giving you validated fields and typed data structures.


In [8]:
import json

# Parse the structured JSON response
audit_data = json.loads(response.text)
report = VulnerabilityAuditReport.model_validate(audit_data)

print(f"=== {report.project_name.upper()} ===")
print(f"Summary: {report.summary}\n")
print(f"Identified {len(report.issues)} issue(s):")
print("-" * 70)

for idx, issue in enumerate(report.issues, 1):
    print(f"[{idx}] {issue.title} ({issue.cwe_id}) - Severity: {issue.severity}")
    print(f"    Affected Files : {', '.join(issue.affected_files)}")
    print(f"    Root Cause     : {issue.root_cause_analysis}")
    print(f"    Explanation    : {issue.vulnerability_explanation[:200]}...")
    print("-" * 70)


=== MICRO-BANKING-SERVICE ===
Summary: High-risk architectural race condition (TOCTOU) identified across transfer_api.py and session_store.py allowing concurrent double-spending under asynchronous load.

Identified 1 issue(s):
----------------------------------------------------------------------
[1] Non-Atomic Idempotency Verification and State Update (CWE-367) - Severity: High
    Affected Files : transfer_api.py, session_store.py
    Root Cause     : Decoupled check-then-act pattern across service boundaries without concurrency primitives.
    Explanation    : Concurrent async requests sharing an identical idempotency_key can simultaneously pass the is_idempotency_key_active check before either thread invokes mark_key_processed...
----------------------------------------------------------------------


## Apply & Verify Synthesized Unified Diff Patch

Inspect the generated patch and see how an automated CI/CD pipeline applies and verifies it programmatically.

A unique capability of deep reasoning is generating valid, syntactically correct unified diffs (`diff -u`) that account for cross-file imports and thread safety primitives.


In [9]:
# Display the generated unified diff patch
first_issue = report.issues[0]
print("Generated Unified Diff Patch:")
print("=" * 60)
print(first_issue.unified_diff_patch)
print("=" * 60)


Generated Unified Diff Patch:
--- a/session_store.py
+++ b/session_store.py
@@ -2,6 +2,7 @@
 import time
+import threading
 from typing import Dict, Any, Optional
 
 class SessionStore:
     def __init__(self):
         self._sessions: Dict[str, Dict[str, Any]] = {}
         self._processed_keys: Dict[str, float] = {}
+        self._lock = threading.Lock()
 
+    def try_claim_idempotency_key(self, key: str) -> bool:
+        with self._lock:
+            ts = self._processed_keys.get(key)
+            if ts and (time.time() - ts) < 60.0:
+                return False
+            self._processed_keys[key] = time.time()
+            return True


### Programmatic Verification

Verify how the proposed changes enhance thread safety by reviewing the remediation:
* Introduces atomic check-and-set concurrency locks (`threading.Lock`) in `SessionStore`.
* Ensures idempotency registration is atomic with balance verification, eliminating the TOCTOU race condition.


In [10]:
# Inspect verification guidance provided by Gemini 4 Argon
print("Automated Verification Steps Recommended by Gemini 4 Argon:")
for i, step in enumerate(report.verification_steps, 1):
    print(f"{i}. {step}")


Automated Verification Steps Recommended by Gemini 4 Argon:
1. Spin up 10 concurrent async worker threads using the same idempotency key and user balance.
2. Assert that exactly 1 request succeeds and 9 requests receive 'Duplicate transaction detected'.
3. Verify final account balance reflects exactly one deduction.


## Conclusion & Key Takeaways

In this guide, you explored how **Gemini 4 Argon** with tunable reasoning enables automated, multi-file software security auditing:

1. **Tunable Reasoning (`thinking_level`):** Higher reasoning levels allow the model to unpack non-trivial architectural bugs (such as TOCTOU race conditions) across file boundaries.
2. **Thought Signature Inspection:** Examining `include_thoughts=True` provides transparency and auditable proof of how the model deduced the flaw.
3. **Structured Pydantic Outputs:** Using `response_schema` eliminates parsing errors and integrates directly with security scanners and automated PR generators.
4. **Actionable Remediation:** Combining deep reasoning with unified diff generation enables automated security fixes in modern developer toolchains.

### Next Steps & Further Reading
* Explore the [Gemini API Thinking Documentation](https://ai.google.dev/gemini-api/docs/thinking) for detailed guidance on reasoning budgets.
* Try integrating this pattern into a GitHub Action workflow to automatically audit pull requests on code change events!
